# SECTION 08: Inventory Analysis

## Objective

In this section, we will analyze product inventory levels, stock availability, and warehouse distribution to identify inventory trends, optimize stock management, and support operational decision-making.

In [14]:
--Before going to perform the analysis, make sure to select the database in which you want to run the query.
/*Required tables:
    Production.Product
    production.productinventory
    Production.Location
*/

Commands completed successfully.

Total execution time: 00:00:00.004

In [ ]:
--Query.1

print 'Products with Low Stock';

with lowstocks as
(
    select p.productid,
        p.name,
        SUM(pi.quantity) as stockcount
    from production.product p
    join production.productinventory pi
        on p.productid = pi.productid
    group by p.productid, p.name
)

select *from lowstocks
order by stockcount asc
go

--Querry.2

print 'invenory Distribution by Location';

with location_distribution as
(
    select l.locationid,
        l.name as locationname,
        SUM(pi.quantity) as stockcount,
        count(distinct pi.productid) as productcount
    from production.location l    
    join production.productinventory pi
        on l.locationid = pi.locationid
    group by l.locationid, l.name
)
select *from location_distribution
order by stockcount desc;


--Querry.3

print 'Hightest stock product at eaach location';


with Hightest_product as

(
    select l.name as locationname,
        p.name as productname,
        SUM(pi.quantity) as stockcount
        from production.location l
        join production.productinventory pi
            on l.locationid = pi.locationid
        join production.product p on p.productid = pi.productid
        group by l.name, p.name
),
ranked_products as (
    select *,dense_rank() over(partition by locationname order by stockcount desc) as rank  from Hightest_product
)

select *from ranked_products
where rank = 1



# SECTION 09: Executive Dashboard

## Objective

In this section, we will build a single comprehensive SQL query that combines key business KPIs such as total revenue, total orders, total customers, average order value, best-selling product, and best-performing territory into one executive-level dashboard result.

In [ ]:
--Before going to perform the analysis, make sure to select the database in which you want to run the query.
/*Required tables:
    sales.salesorderheader	
    sales.salesorderdetail	
    production.product	
    sales.salesterritory
*/

In [16]:

--Key Performance Indicators (KPIs) 
--What is the most  effective product and territory in terms of revenue generation?
--This will give us an idea about which product and territory is generating the most revenue for the company.

with kpis as
(
    select
        sum(totaldue) as total_revenue,
        count(salesorderid) as total_orders,
        count(distinct customerid) as total_customers,
        avg(totaldue) as average_order_value
    from sales.salesorderheader
),

best_product as
(
    select top 1
        p.name as best_selling_product
    from sales.salesorderdetail s
    join production.product p
        on s.productid = p.productid
    group by
        p.productid,
        p.name
    order by sum(s.linetotal) desc
),

best_territory as
(
    select top 1
        t.name as best_territory
    from sales.salesorderheader h
    join sales.salesterritory t
        on h.territoryid = t.territoryid
    group by
        t.territoryid,
        t.name
    order by sum(h.totaldue) desc
)

select
    o.total_revenue,
    o.total_orders,
    o.total_customers,
    o.average_order_value,
    p.best_selling_product,
    t.best_territory
from kpis o
cross join best_product p
cross join best_territory t;
go

(1 row affected)

total_revenue  | total_orders | total_customers | average_order_value | best_selling_product   | best_territory
---------------+--------------+-----------------+---------------------+------------------------+---------------
123216786.1159 | 31465        | 19119           | 3915.9951           | Mountain-200 Black, 38 | Southwest     
(1 row)

Total execution time: 00:00:00.325

# SECTION 10: Advanced SQL Concepts

## Objective

In this section, we will apply advanced SQL features such as CTEs, Views, Stored Procedures, Scalar Functions, Inline Table-Valued Functions, and Temporary Tables to create reusable and business-oriented analytical solutions.

In [ ]:
--Query.1

/* The management team wants to identify customers whose total spending is above the average customer*/

with customer_spending as
(
    select
        c.customerid,
        p.firstname + ' ' + p.lastname as customername,
        sum(h.totaldue) as total_spending
    from sales.salesorderheader h
    join sales.customer c
        on h.customerid = c.customerid
    join person.person p
        on c.personid = p.businessentityid
    group by
        c.customerid,
        p.firstname,
        p.lastname
),
average_spending as
(
    select avg(total_spending) as average_spending
    from customer_spending
)

select
    cs.customerid,
    cs.customername,
    cs.total_spending
from customer_spending cs
cross join average_spending a
where cs.total_spending > a.average_spending
order by cs.total_spending desc;
GO

(5381 rows affected)

customerid | customername        | total_spending
-----------+---------------------+---------------
16699      | Angel Adams         | 74.6759       
16691      | Carlos Adams        | 4941.2728     
16902      | Eric Adams          | 234.2269      
16730      | Evan Adams          | 155.7388      
16656      | Hunter Adams        | 6552.9042     
16910      | Jackson Adams       | 156.8659      
16828      | James Adams         | 7936.8394     
16754      | Jesse Adams         | 3446.0531     
16831      | Jonathan Adams      | 7963.0389     
16846      | Kevin Adams         | 199.9498      
16837      | Logan Adams         | 171.7944      
16734      | Mason Adams         | 142.5119      
16840      | Nathan Adams        | 195.5298      
16662      | Noah Adams          | 4813.0596     
16638      | Samuel Adams        | 6601.5242     
12041      | Alisha Alan         | 6628.0442     
12927      | Aidan Alexander     | 119.8484      
20216      | Alyssa Alexande

In [ ]:
--Query.2 --View


print 'Management frequently needs a report showing product sales performance, so create a reusable view.';
go

create view sales_report
as 
select p.productid,
        p.name,
        sum(o.orderqty) as total_quantity_sold,
        sum(o.linetotal) as total_revenue
from production.product p  
join sales.salesorderdetail  o 
        on o.productid=p.productid
group by 
    p.productid,
    p.name;
GO



Management frequently needs a report showing product sales performance, so create a reusable view.

  CREATE_VIEW - dbo.sales_report

Total execution time: 00:00:00.168

In [ ]:
--querry.3

PRINT 'The sales manager wants to generate a sales report for any year without rewriting the query each time.';
go

create procedure sales_report_by_year
    @salesyear int
as
begin

    select
        month(orderdate) as month,
        count(salesorderid) as total_orders,
        sum(totaldue) as total_revenue
    from sales.salesorderheader
    where year(orderdate) = @salesyear
    group by month(orderdate)
    order by month(orderdate);

end;
go 

-->>EXEC sales_report_by_year @salesyear=2015;--<<


In [ ]:
--querry.4 --Scaler UDF

print 'The company wants to classify customers based on their total lifetime spending.'
GO

create function customer_category
(
    @customerid int
)
returns varchar(20)
as
begin

    declare @total_spending money;

    select @total_spending = sum(totaldue)
    from sales.salesorderheader
    where customerid = @customerid;

    return
    case
        when @total_spending >= 100000 then 'Platinum'
        when @total_spending >= 50000 then 'Gold'
        when @total_spending >= 10000 then 'Silver'
        else 'Bronze'
    end;

end;
go


In [ ]:
select
    customerid,
    dbo.customer_category(109) as customer_category
from sales.customer;

In [ ]:
--query.5
 
print 'The inventory manager wants to temporarily store the total stock available for each product and then identify the Top 10 highest-stocked products.'
go
create table #product_stock
(
    productid int,
    product_name varchar(50),
    total_stock int
);

insert into #product_stock
select
    p.productid,
    p.name,
    sum(pi.quantity) as total_stock
from production.product p
join production.productinventory pi
    on p.productid = pi.productid
group by
    p.productid,
    p.name;

select top 10
    productid,
    product_name,
    total_stock
from #product_stock
order by total_stock desc;

drop table #product_stock;
go

(432 rows affected)
(10 rows affected)

productid | product_name       | total_stock
----------+--------------------+------------
379       | Hex Nut 7          | 1911       
367       | Thin-Jam Hex Nut 3 | 1901       
383       | Hex Nut 23         | 1901       
387       | Hex Nut 10         | 1888       
393       | Hex Nut 14         | 1880       
489       | Metal Tread Plate  | 1837       
396       | Hex Nut 18         | 1824       
389       | Hex Nut 2          | 1808       
371       | Thin-Jam Hex Nut 7 | 1781       
376       | Hex Nut 6          | 1781       
(10 rows)

Total execution time: 00:00:00.011